# 6.4.4 Text-to-Audio / Speech

Compute a mel spectrogram on a synthetic sine wave signal and perform formant frequency analysis.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
sr = 16000; duration = 1.0
t = np.linspace(0, duration, int(sr*duration), endpoint=False)
signal = (0.6*np.sin(2*np.pi*120*t) + 0.3*np.sin(2*np.pi*800*t) + 0.1*np.sin(2*np.pi*1200*t))
signal *= np.exp(-t*0.5)

freqs = np.fft.rfftfreq(len(signal), 1/sr)
fft_mag = np.abs(np.fft.rfft(signal * np.hanning(len(signal))))
print(f'Signal: {len(signal)} samples, SR={sr}')
peak_idx = np.argsort(fft_mag)[-3:][::-1]
print(f'Top 3 frequency peaks: {freqs[peak_idx].round(1)} Hz')

In [ ]:
n_fft, hop = 512, 256
win = np.hanning(n_fft)
n_frames = (len(signal)-n_fft)//hop + 1
stft = np.zeros((n_fft//2+1, n_frames), dtype=complex)
for i in range(n_frames):
    stft[:,i] = np.fft.rfft(signal[i*hop:i*hop+n_fft] * win)
power = np.abs(stft)**2
log_spec = np.log(power + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t[:800], signal[:800], color='steelblue', lw=1)
axes[0].set(xlabel='Time (s)', ylabel='Amplitude', title='Waveform')
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(log_spec, aspect='auto', origin='lower', cmap='magma')
axes[1].set(xlabel='Frame', ylabel='FFT Bin', title='Log Spectrogram')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('output/mel_spectrogram.png')
print('Saved mel_spectrogram.png')